In [16]:
import pandas as pd
import numpy as np
import os
import re

# 1. Directory and File Paths

In [21]:
raw_dir = "../data/raw/"
processed_dir = "../data/processed/"
os.makedirs(processed_dir, exist_ok=True)

income_file = os.path.join(raw_dir, "ACSST1Y2024.S1901-2026-03-09T222516.csv")
finance_file = os.path.join(raw_dir, "elsec24_sumtables.xlsx")

# 2. Process Census Income Data

In [22]:
df_income_raw = pd.read_csv(income_file)

# Convert wide format to long format
melted = df_income_raw.melt(id_vars=['Label (Grouping)'], var_name='Col', value_name='Val')

# Filter for median income estimates
median_rows = melted[melted['Label (Grouping)'].str.strip() == 'Median income (dollars)'].copy()
est_rows = median_rows[median_rows['Col'].str.contains('!!Households!!Estimate', na=False)].copy()

# Extract clean state names using regular expressions
est_rows['State'] = est_rows['Col'].str.extract(r'^([a-zA-Z\s]+)!!', expand=False).str.strip()

# Clean non-numeric characters from income values and convert to numeric
est_rows['Median_Household_Income'] = est_rows['Val'].replace(r'[^\d.]', '', regex=True)
est_rows['Median_Household_Income'] = pd.to_numeric(est_rows['Median_Household_Income'], errors='coerce')

# Drop duplicates to ensure unique state rows
df_income = est_rows[['State', 'Median_Household_Income']].dropna().drop_duplicates(subset=['State']).reset_index(drop=True)

# 3. Process ELSEC Finance Data

In [23]:
# Read Table 1 directly from the Excel file
df_fin_raw = pd.read_excel(finance_file, sheet_name='Table 1', header=None)

# Identify valid state rows using regex (looking for trailing periods)
state_rows = df_fin_raw[df_fin_raw[0].astype(str).str.contains(r'\.{3,}', na=False)].copy()

# Clean state names by stripping punctuation and extra whitespace
state_rows['State'] = state_rows[0].str.replace(r'[^a-zA-Z\s]', '', regex=True).str.strip()

# Extract total revenue (column 2) and per-pupil spending (column 10)
df_finance = state_rows[['State', 2, 10]].copy()
df_finance.columns = ['State', 'Total_Revenue_Thousands', 'Per_Pupil_Spending']

# Convert extracted columns to numeric
df_finance['Total_Revenue_Thousands'] = pd.to_numeric(df_finance['Total_Revenue_Thousands'], errors='coerce')
df_finance['Per_Pupil_Spending'] = pd.to_numeric(df_finance['Per_Pupil_Spending'], errors='coerce')

# Filter out national aggregate rows
df_finance = df_finance[df_finance['State'] != 'Reporting Areas'].reset_index(drop=True)

# 4. Record Linkage / Data Integration

In [ ]:
# Merge datasets directly on the "State" key
master_df = pd.merge(df_income, df_finance, on="State", how="left")

# Drop rows without a valid state name
master_df = master_df.dropna(subset=['State'])

# Save the integrated dataset
output_path = os.path.join(processed_dir, "integrated_special_ed_data.csv")
master_df.to_csv(output_path, index=False)

print(f"Dataset shape: {master_df.shape}")
print("First 5 rows of integrated data:")
print(master_df.head().to_string())
print(f"\nSaved integrated data to: {output_path}")

Dataset shape: (52, 4)
First 5 rows of integrated data:
        State  Median_Household_Income  Total_Revenue_Thousands  Per_Pupil_Spending
0     Alabama                    66659               12853947.0        13598.057975
1      Alaska                    95665                      NaN                 NaN
2     Arizona                    81486               13858937.0        12002.958476
3    Arkansas                    62106                7274695.0        13872.571401
4  California                   100149              135520606.0        20233.333934

Saved integrated data to: ../data/processed/integrated_special_ed_data.csv
